# Installation et importation des bibliothèques

In [ ]:
import time
print("---INSTALLATION DES BIBLIOTHEQUES EN COURS...---")
start_time = time.time()
!pip install -q monai[nibabel,pillow,itk]
finish_time = time.time()
print(f"INSTALLATION DES BIBLIOTHEQUES FINIE EN {finish_time - start_time:.2f} SECONDES")

---INSTALLATION DES BIBLIOTHEQUES EN COURS...---
INSTALLATION DES BIBLIOTHEQUES FINIE EN 11.76 SECONDES


In [ ]:
print("---IMPORTATION DES BIBLIOTHEQUES EN COURS...---")
start_time = time.time()
import glob
import itertools
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import os
import random
import scipy.ndimage as ndimage

import torch
import torch.nn as nn

from google.colab import drive
from monai.data import CacheDataset, DataLoader, Dataset, decollate_batch
from monai.losses import DiceLoss
from monai.metrics import DiceMetric, HausdorffDistanceMetric, SurfaceDistanceMetric
from monai.networks.nets import UNet
from monai.networks.utils import one_hot
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, ScaleIntensityd, Resized, EnsureTyped, CopyItemsd, RandAffined, AsDiscrete, KeepLargestConnectedComponent, RandGaussianSmoothd
)

from skimage.measure import label, regionprops, perimeter
from skimage.morphology import binary_erosion

from tqdm.notebook import tqdm

finish_time = time.time()
print(f"IMPORTATION DES BIBLIOTHEQUES FINIE EN {finish_time - start_time:.2f} SECONDES")

---IMPORTATION DES BIBLIOTHEQUES EN COURS...---
IMPORTATION DES BIBLIOTHEQUES FINIE EN 0.00 SECONDES


# Connection au Google Drive

In [ ]:
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
base_projet = '/content/drive/MyDrive/TFE_Segmentation'
root_dir = os.path.join(base_projet, 'Datasets', 'Task02_Heart')

MessageError: Error: credential propagation was unsuccessful

# Scan des patients et vérification des formats

In [ ]:
print("---SCAN DES PATIENTS EN COURS...---")
start_time = time.time()
images = sorted(glob.glob(os.path.join(root_dir, "imagesTr", "*.nii.gz")))
labels = sorted(glob.glob(os.path.join(root_dir, "labelsTr", "*.nii.gz")))

data = [{"image": img, "label": lbl} for img, lbl in zip(images, labels) if os.path.exists(img) and os.path.exists(lbl)]
print(f"Nombre de patients Décathlon : {len(data)}")

# Découpage 4/5 entrainement et 1/5 validation
split_idx = int(len(data) * 0.8)
train_data = data[:split_idx]
val_data = data[split_idx:]


finish_time = time.time()
print(f"Patients d'entrainement : {len(train_data)}")
print(f"Patients de validation : {len(val_data)}")
print(f"SCAN DES PATIENTS FINI EN {finish_time - start_time:.2f} SECONDES")

In [ ]:
# On sélectionne le premier patient pour l'analyse
premier_patient = data[0]
print("==================================================")
print(f"🔍 ANALYSE DU PATIENT DE RÉFÉRENCE : {os.path.basename(premier_patient['image'])}")
print("==================================================")

# 1. INSPECTION DES FICHIERS BRUTS (SUR LE DISQUE VIA NIBABEL)
img_brute = nib.load(premier_patient["image"])
lbl_brute = nib.load(premier_patient["label"])

print("\n📁 [1/2] Dimensions et métadonnées des fichiers bruts sur disque :")
print(f"  -> Image brute 3D - Dimensions : {img_brute.shape}")
print(f"  -> Label brut 3D - Dimensions : {lbl_brute.shape}")

# Extraction de l'espacement physique des voxels (spacing en mm)
spacing_img = img_brute.header.get_zooms()
spacing_lbl = lbl_brute.header.get_zooms()
print(f"  -> Résolution physique de l'image (Voxel Spacing) : {spacing_img[0]:.2f} x {spacing_img[1]:.2f} x {spacing_img[2]:.2f} mm")
print(f"  -> Résolution physique du label : {spacing_lbl[0]:.2f} x {spacing_lbl[1]:.2f} x {spacing_lbl[2]:.2f} mm")

# Pipeline de transformation (profil basses calories pour CPU)

In [ ]:
transforms = Compose([
    # Chargemebt des volumes NiFTI (.nii.gz)
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),

    # Normalisation du contraste médical (0 à 1)
    ScaleIntensityd(keys="image"),

    # Duplication de l'image native
    # Servira de Verité terrain pour le SRCNN et de support pour la segmentation
    CopyItemsd(keys=["image"], names=["image_hr"]),

    # Simulation d'une dégradation IRM réaliste sur la clé 'image' qui devient la LR
    # Application d'un flou pour casser les hautes fréquences avant de réduire la taille
    RandGaussianSmoothd(
        keys="image",
        sigma_x=(1.0, 1.0),
        sigma_y=(1.0, 1.0),
        sigma_z=(0.0, 0.0),
        prob=1.0
    ),

    # Redimensionnement du label en haute definition (256x256) avec nearest pour éviter la création de fausses classes interpolées
    Resized(keys=["image"], spatial_size=[128, 128, 32], mode=['bilinear']),
    Resized(keys=["image_hr"], spatial_size=[256, 256, 32], mode=['bilinear']),
    Resized(keys=["label"], spatial_size=[256, 256, 32], mode=['nearest']),

    EnsureTyped(keys=["image", "image_hr", "label"]),
])

In [ ]:
# Utilisation d'un Dataset standard (sans Cache) pour évaluer un seul élément à la volée
print("\n⚙️ [2/2] Application du pipeline de transformation...")
dataset_simple = Dataset(data=[premier_patient], transform=transforms)

# Chargement et évaluation de l'échantillon transformé
echantillon_transforme = dataset_simple[0]

print("\n📊 Dimensions finales des tenseurs PyTorch obtenus :")
print(f"  -> Clé 'image' (Basse Résolution pour SRCNN) : {echantillon_transforme['image'].shape} (Type: {echantillon_transforme['image'].dtype})")
print(f"  -> Clé 'label' (Masque Haute Résolution pour Loss Dice) : {echantillon_transforme['label'].shape} (Type: {echantillon_transforme['label'].dtype})")
print("==================================================")

# Mise en cache total en RAM

In [ ]:
print(f"---CHARGEMENT DU DATASET EN COURS...---")
start_time = time.time()

# Extraction des tranches 2D dans des listes statiques avant de les envoyer au DataLoader
train_ds = CacheDataset(data=train_data, transform=transforms, cache_rate=1.0, num_workers=2)
val_ds = CacheDataset(data=val_data, transform=transforms, cache_rate=1.0, num_workers=2)
finish_time = time.time()
print(f"\nCHARGEMENT DU DATASET FINI EN {finish_time - start_time:.2f} SECONDES")

In [ ]:
# Fonction d'extraction de tranches uniquement pour la segmentation 2D
def extraire_tranches(dataset):
    tranches = []
    for data in dataset:
        image_lr = data["image"] # Basse résolution
        image_hr = data["image_hr"] # Haute résolution
        label = data["label"] # Masque de segmentation

        # Convertir le label en numpy pour scanner l'axe Z
        volume_label = label.squeeze().cpu().numpy() if hasattr(label, 'cpu') else label.squeeze()

        # Parcourir chaque coupe sur l'axe Z
        num_slices = volume_label.shape[-1]
        for z in range(num_slices):
          # Extraction de la tranche si l'atrium est visible
          if (volume_label[..., z] > 0).any():
            tranches.append({
                "image_lr": image_lr[..., z].clone(), # Entrée de la SR
                "image_hr": image_hr[..., z].clone(), # Cible de la SR
                "label": label[..., z].clone() # Cible de la segmentation
            })
    if len(tranches) == 0: # Aucune tranche trouvée : on prend par défaut la centrale
      print("Aucune tranche trouvée dans le dataset.")
      for data in dataset:
        mid_slice = data["image"].shape[-1] // 2
        tranches.append({
          "image_lr": data["image"][..., mid_slice].clone(),
          "image_hr": data["image_hr"][..., mid_slice].clone(),
          "label": data["label"][..., mid_slice].clone()
        })
    return tranches

In [ ]:
print("---EXTRACTION DES TRANCHES EN COURS...---")
start_time = time.time()
# Extraction des tranches 2D dans des listes statiques avant de les envoyer au DataLoader
train_dataset_2d = extraire_tranches(train_ds)
val_dataset_2d = extraire_tranches(val_ds)
train_loader = DataLoader(train_dataset_2d, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset_2d, batch_size=2, shuffle=False)
finish_time = time.time()
print(f"EXTRACTION DES TRANCHES FINIE EN {finish_time - start_time:.2f} SECONDES")

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# Définition des différentes métriques

In [ ]:
loss_function = DiceLoss(sigmoid=True, include_background=True)

assd_metric = SurfaceDistanceMetric(include_background=False, reduction="mean")
hd95_metric = HausdorffDistanceMetric(include_background=False, percentile=95, reduction="mean")
dice_metric = DiceMetric(include_background=False, reduction="mean")

post_binarize = AsDiscrete(threshold=0.5) # Binarisation à 0.5
keep_largest = KeepLargestConnectedComponent(applied_labels=[1]) # Conservation de la plus grande composante (sur le canal/classe de l'atrium)

nb_epochs = 100

# Fonction d'entrainement et de validation du modèle

In [ ]:
def entrainement_model(num_cas, unet_model, srcnn_model, optimizer, nom_fichiers):
  print(f"---ENTRAINEMENT DU CAS {num_cas} SUR {nb_epochs} EPOQUES EN COURS...---")

  historique_trainloss, historique_valloss, historique_dice, historique_hausdorff, historique_assd = [], [], [], [], []
  meilleur_dice = 0
  start_time = time.time()

  chemin_unet = os.path.join(base_projet, nom_fichiers[0])
  chemin_srcnn = os.path.join(base_projet, nom_fichiers[1]) if nom_fichiers[1] != '' else None

  for epoch in range(nb_epochs):
      unet_model.train()
      if num_cas == 4 and srcnn_model is not None:
        srcnn_model.train()
      elif srcnn_model is not None:
        srcnn_model.eval()

      running_loss = 0

      for batch_data in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{nb_epochs}", leave=False):
          targets_hr = batch_data["image_hr"].to(device) # Tenseur de taille (256x256)
          inputs_lr = batch_data["image_lr"].to(device) # Tenseur de taille (128x128)
          inputs_labels = batch_data["label"].to(device) # Cible finale pour le UNET

          inputs_upsampled = nn.functional.interpolate(inputs_lr, size=(256,256), mode='bilinear', align_corners=False)

          if num_cas == 1: # Baseline HR Native
            outputs = unet_model(targets_hr)
            loss = loss_function(outputs, inputs_labels)
          elif num_cas == 2: # Baseline LR (interpolation bilinéaire simple)
            outputs = unet_model(inputs_upsampled)
            loss = loss_function(outputs, inputs_labels)
          elif num_cas == 3: # Séquentiel (SRCNN pré-entrainé figé + U-Net)
            with torch.no_grad():
              inputs_sr = srcnn_model(inputs_upsampled)
            outputs = unet_model(inputs_sr.detach())
            loss = loss_function(outputs, inputs_labels)
          elif num_cas == 4: # Apprentissage conjoint (end-to-end)
            inputs_sr = srcnn_model(inputs_upsampled)
            outputs = unet_model(inputs_sr)

            loss_reconstruction = nn.functional.mse_loss(inputs_sr, targets_hr)
            loss_segmentation = loss_function(outputs, inputs_labels)

            loss = 0.8 * loss_segmentation + 0.2 * loss_reconstruction
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

          running_loss += loss.item()

      # Calcul de la perte d'entrainement moyenne pour l'époch
      train_loss = running_loss/len(train_loader)

      # Evaluation sur la partie validation
      val_loss, dice_score, hausdorff_score, assd_score = validation_model(num_cas, unet_model, srcnn_model)

      # Stockage des scores
      historique_trainloss.append(train_loss)
      historique_valloss.append(val_loss)
      historique_dice.append(dice_score)
      historique_hausdorff.append(hausdorff_score)
      historique_assd.append(assd_score)

      if dice_score > meilleur_dice:
        meilleur_dice = dice_score
        torch.save(unet_model.state_dict(), os.path.join(base_projet, '2D', f'model{num_cas}decathlon.pth'))
        if num_cas == 4 and srcnn_model is not None:
          torch.save(srcnn_model.state_dict(), os.path.join(base_projet, '2D', f'srcnn{num_cas}decathlon.pth'))

      if (epoch+1) == 1 or (epoch+1) % 10 == 0 or (epoch+1) == nb_epochs:
        print(f"Epoch {epoch + 1}/{nb_epochs} / Loss(Train/Val):{train_loss:.4f}/{val_loss:.4f} / Dice: {dice_score:.4f} / HD95 : {hausdorff_score:.4f} / ASSD : {assd_score:.4f}" )
  print(f"ENTRAINEMENT TERMINE EN {time.time()-start_time:.2f} SECONDES\n")
  resultats = {'Train Loss' : historique_trainloss, 'Val Loss': historique_valloss, 'Dice': historique_dice, 'HD95': historique_hausdorff, 'ASSD': historique_assd}
  return resultats


def validation_model(num_cas, unet_model, srcnn_model):
  unet_model.eval()
  if srcnn_model is not None:
    srcnn_model.eval()

  epoch_loss = 0
  with torch.no_grad():
    for val_data in val_loader:
      val_targets_hr = val_data["image_hr"].to(device)
      val_inputs_lr = val_data["image_lr"].to(device)
      val_labels = val_data["label"].to(device)

      val_upsampled = nn.functional.interpolate(val_inputs_lr, size=(256,256), mode='bilinear', align_corners=False)

      if num_cas == 1: # Baseline HR Native
        val_input = val_targets_hr
      elif num_cas == 2: # Baseline LR Bilinéaire
        val_input = val_upsampled
      elif num_cas in (3, 4): # Pipelined / Conjoint SRCNN -> U-Net
        val_input = srcnn_model(val_upsampled)

      # Prédiction UNet
      val_outputs = unet_model(val_input)
      loss_dice = loss_function(val_outputs, val_labels)
      epoch_loss += loss_dice.item()

      # Post-traitement, binarisation
      val_outputs_prob = torch.sigmoid(val_outputs)
      val_outputs_bin = post_binarize(val_outputs_prob)
      val_outputs_clear = keep_largest(val_outputs_bin)

      dice_metric(y_pred=val_outputs_clear, y=val_labels)
      hd95_metric(y_pred=val_outputs_clear.cpu(), y=val_labels.cpu())
      assd_metric(y_pred=val_outputs_clear.cpu(), y=val_labels.cpu())

  # Calcul de la perte moyenne
  epoch_loss /= len(val_loader)
  # Récupération des scores agrégés
  dice_score = dice_metric.aggregate().item()
  # Modification pour avoir un tenseur et non une liste
  hausdorff_agg = hd95_metric.aggregate()
  assd_agg = assd_metric.aggregate()

  # Extraction et conversion Numpy. Si Monai renvoie un tenseur Python, on le détache sinon on le convertit
  if hasattr(hausdorff_agg, 'cpu'):
    hd95_np = hausdorff_agg.cpu().detach().numpy().flatten()
  else:
    hd95_np = np.array(hausdorff_agg).flatten()
  if hasattr(assd_agg, 'cpu'):
    assd_np = assd_agg.cpu().detach().numpy().flatten()
  else:
    assd_np = np.array(assd_agg).flatten()

  # Conversion en float numérique pour éliminer les textes/vides/Nan de Monai
  hd95_np = pd.to_numeric(hd95_np, errors='coerce')
  assd_np = pd.to_numeric(assd_np, errors='coerce')

  # Filtrage : garder uniquement les valeurs différentes de inf/nan
  hd95_valides = hd95_np[np.isfinite(hd95_np)]
  assd_valides = assd_np[np.isfinite(assd_np)]

  # Calcul moyenne uniquement sur les tranches mesures (+ valeur de sécurité si le modèle est rempli de inf écran noir)
  hausdorff_score = float(np.mean(hd95_valides)) if len(hd95_valides) > 0 else 50.0
  assd_score = float(np.mean(assd_valides)) if len(assd_valides) > 0 else 50.0

  dice_metric.reset()
  hd95_metric.reset()
  assd_metric.reset()

  return epoch_loss, dice_score, hausdorff_score, assd_score

# Fonction d'initialisation du modèle 2D UNET

In [ ]:
def creation_model():
  unet_model = UNet(
    spatial_dims=2,
    in_channels=1,
    out_channels=1, #Binaire : Atrium ou non
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    dropout=0.2,
  ).to(device)
  return unet_model

# Définition de l'architecture SRCNN

In [ ]:
class SRCNN(nn.Module):
  def __init__(self):
    super(SRCNN, self).__init__()
    self.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=9, padding=9 // 2)
    self.conv2 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=5, padding=5 // 2)
    self.conv3 = nn.Conv2d(in_channels=32, out_channels=1, kernel_size=5, padding=5 // 2)
    self.relu = nn.ReLU()

  def forward(self, x):
    # Sauvegarde de l'image d'entrée (upsamplée floue de taille 256x256)
    identity = x
    # Passage à travers les couches de convolution du SRCNN
    x = self.relu(self.conv1(x))
    x = self.relu(self.conv2(x))
    res = self.conv3(x) # Extraction des détails manquants
    return identity + res # Addition de l'image floue et des détails fins reconstruits par l'IA

# Entrainement baseline

In [ ]:
unet_model1 = creation_model()
optimizer1 = torch.optim.Adam(unet_model1.parameters(), lr=0.001, weight_decay=1e-5) # Mise à jour des paramètres uniquement du UNet
resultats1 = entrainement_model(1, unet_model1, None, optimizer1, ('model1.pth', ''))

# Séquentiel Aval (Unet -> SRCNN)

In [ ]:
# Initialisation du modèle UNet pour le SRCNN solo
unet_model2 = creation_model()
optimizer2 = torch.optim.Adam(unet_model2.parameters(), lr=0.001, weight_decay=1e-5) # Mise à jour des paramètres uniquement du UNet
resultats2 = entrainement_model(2, unet_model2, None, optimizer2, ('model2.pth', ''))

# Séquentiel amont (SRCNN -> U-Net)

In [ ]:
srcnn_solo = SRCNN().to(device)
optimizer_solo = torch.optim.Adam(srcnn_solo.parameters(), lr=0.001, weight_decay=1e-5) # Mise à jour des paramètres uniquement du SRCNN
mse_loss = nn.MSELoss()

for epoch in range(100):
  srcnn_solo.train()
  for batch_data in tqdm(train_loader, desc=f"Epoch {epoch + 1}/100", leave=False):
    targets_hr = batch_data["image_hr"].to(device) # Tenseur de taille (256x256)
    inputs_lr = batch_data["image_lr"].to(device) # Tenseur de taille (128x128)
    inputs_upsampled = nn.functional.interpolate(inputs_lr, size=(256,256), mode='bilinear', align_corners=False)

    optimizer_solo.zero_grad()
    outputs = srcnn_solo(inputs_upsampled)
    loss = mse_loss(outputs, targets_hr)
    loss.backward()
    optimizer_solo.step()

torch.save(srcnn_solo.state_dict(), os.path.join(base_projet, '2D', 'srcnn_isole.pth'))
srcnn_solo.eval()

# Initialisation du modèle UNet pour le SRCNN solo
unet_model3 = creation_model()
optimizer3 = torch.optim.Adam(unet_model3.parameters(), lr=0.001, weight_decay=1e-5) # Mise à jour des paramètres uniquement du UNet
resultats3 = entrainement_model(3, unet_model3, srcnn_solo, optimizer3, ('model3.pth', ''))

# Entrainement conjoint End-to-End

In [ ]:
# Réinstanciation d'un SRCNN et d'un modèle neuf
unet_model4 = creation_model()
srcnn_joint = SRCNN().to(device)
# Un seul optimiseur pour mettre à jour le SRCNN ET le modèle
optimizer4 = torch.optim.Adam(
    list(srcnn_joint.parameters()) + list(unet_model4.parameters()),
    lr=1e-3
)
resultats4 = entrainement_model(4, unet_model4, srcnn_joint, optimizer4, ('model4.pth', 'srcnn4.pth'))

# Sauvegarde des métriques

In [ ]:
df_historique = pd.DataFrame({
    'Epoch': range(1, nb_epochs + 1),
    'Train Loss1' : resultats1['Train Loss'], 'Val Loss1': resultats1['Val Loss'], 'Dice1': resultats1['Dice'], 'HD95_1': resultats1['HD95'], 'ASSD1': resultats1['ASSD'],
    'Train Loss2' : resultats2['Train Loss'], 'Val Loss2': resultats2['Val Loss'], 'Dice2': resultats2['Dice'], 'HD95_2': resultats2['HD95'], 'ASSD2': resultats2['ASSD'],
    'Train Loss3' : resultats3['Train Loss'], 'Val Loss3': resultats3['Val Loss'], 'Dice3': resultats3['Dice'], 'HD95_3': resultats3['HD95'], 'ASSD3': resultats3['ASSD'],
    'Train Loss4' : resultats4['Train Loss'], 'Val Loss4': resultats4['Val Loss'], 'Dice4': resultats4['Dice'], 'HD95_4': resultats4['HD95'], 'ASSD4': resultats4['ASSD']
})

df_historique.to_csv(os.path.join(base_projet, '2D', 'resultats_decathlon.csv'), index=False)
print("Résultats sauvegardés avec succès")

# Calcul score de stabilité

In [ ]:
# Calcul de la stabilité sur les 50 dernières époques
stabilites = {
    'Cas 1 (Baseline HR)': df_historique['Dice1'].tail(50).std(),
    'Cas 2 (SRCNN Solo -> U-Net)': df_historique['Dice2'].tail(50).std(),
    'Cas 3 (Interpolation -> U-Net)': df_historique['Dice3'].tail(50).std(),
    'Cas 4 (Conjoint End-To-End)': df_historique['Dice4'].tail(50).std()
}

print("Stabilité du Dice score sur les 50 dernières époques :")
for item, valeur in stabilites.items():
  print(f"{item} : {valeur:.4f}")

# Génération des graphiques

In [ ]:
def lisser_graph(donnees, fenetre=5):
  return pd.Series(donnees).rolling(window=fenetre, min_periods=1, center=True).mean()

total_epochs = range(1, nb_epochs + 1)

plt.figure(figsize=(15, 10))

# Graphique des Scores
plt.subplot(1,2,1)
plt.plot(total_epochs, lisser_graph(resultats1['Dice']), label='U-Net', linestyle='--', color='blue')
plt.plot(total_epochs, lisser_graph(resultats2['Dice']), label='SRCNN -> U-Net', color='green')
plt.plot(total_epochs, lisser_graph(resultats3['Dice']), label='Interpolation -> U-Net', color='red')
plt.plot(total_epochs, lisser_graph(resultats4['Dice']), label='U-Net + SRCNN Joint', linewidth=2, color='orange')
plt.xlabel('Époques')
plt.ylabel('Dice Score')
plt.title('Comparaison Dice Score sur 200 époques sur 4 cas')
plt.legend()
plt.grid(True)

# Graphique de Hausdorff
plt.subplot(1,2,2)
plt.plot(total_epochs, lisser_graph(resultats1['HD95']), label='U-Net', linestyle='--', color='blue')
plt.plot(total_epochs, lisser_graph(resultats2['HD95']), label='SRCNN -> U-Net', color='green')
plt.plot(total_epochs, lisser_graph(resultats3['HD95']), label='Interpolation -> U-Net', color='red')
plt.plot(total_epochs, lisser_graph(resultats4['HD95']), label='U-Net + SRCNN Joint', linewidth=2, color='orange')
plt.xlabel('Époques')
plt.ylabel('Hausdorff Distance (mm)')
plt.title('Comparaison Hausdorff Distance sur 200 époques sur 4 cas')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(base_projet, '2D', 'graphiques.png'), dpi=300)
plt.show()

# Comparaison visuelle

In [ ]:
def afficher_comparaison_visuelle(idx_echantillon):
  # Récupération d'un échantillon de val_loader
  val_iter = iter(val_loader)
  batch = next(val_iter)

  # Vérification du nombre d'éléments dans le lot
  batch_size = batch["image_hr"].shape[0]
  if idx_echantillon >= batch_size:
    idx_echantillon = 0
    print("Index hors bornes")

  # Extraction de l'image et du masque de vérité terrain (Ground Terrain)
  img_hr = batch["image_hr"][idx_echantillon:idx_echantillon+1].to(device)
  img_lr = batch["image_lr"][idx_echantillon:idx_echantillon+1].to(device)
  label_gt = batch["label"][idx_echantillon:idx_echantillon+1].to(device)

  # On évalue tous les modèles
  unet_model1.eval()
  unet_model2.eval()
  unet_model3.eval()
  unet_model4.eval()
  if srcnn_solo is not None: srcnn_solo.eval()
  if srcnn_joint is not None: srcnn_joint.eval()

  with torch.no_grad():
    # Prédiction du modèle 1 (vraie HR native du DataLoader)
    output_model1 = unet_model1(img_hr)
    bin_model1 = post_binarize(torch.sigmoid(output_model1))
    pred_model1 = keep_largest(bin_model1)

    # Récupération de la LR upsamplée du DataLoader. On donne input_upsampled aux SRCNN dans la boucle d'entrainement
    img_upsampled = nn.functional.interpolate(img_lr, size=(256,256), mode='bilinear', align_corners=False)

    # Prédiction du modèle 2
    output_model2 = unet_model2(img_upsampled)
    bin_model2 = post_binarize(torch.sigmoid(output_model2))
    pred_model2 = keep_largest(bin_model2)

    # Prédiction du modèle 3
    img_srcnn_solo = srcnn_solo(img_upsampled)
    output_model3 = unet_model3(img_srcnn_solo)
    bin_model3 = post_binarize(torch.sigmoid(output_model3))
    pred_model3 = keep_largest(bin_model3)

    # Prédiction du modèle 4
    img_srcnn_joint = srcnn_joint(img_upsampled)
    output_model4 = unet_model4(img_srcnn_joint)
    bin_model4 = post_binarize(torch.sigmoid(output_model4))
    pred_model4 = keep_largest(bin_model4)

  # Conversion des tenseurs en Numpy pour matplotlib
  img_display = img_hr[0, 0].cpu().numpy()
  label_display = label_gt[0, 0].cpu().numpy()
  pred_model1_display = pred_model1[0, 0].cpu().numpy()
  pred_model2_display = pred_model2[0, 0].cpu().numpy()
  pred_model3_display = pred_model3[0, 0].cpu().numpy()
  pred_model4_display = pred_model4[0, 0].cpu().numpy()

  # Affichage des images
  plt.figure(figsize=(15, 5))

  # Image d'origine (HR)
  plt.subplot(2, 3, 1)
  plt.imshow(img_display, cmap='gray')
  plt.title('Image originale')
  plt.axis('off')

  # Masque de vérité terrain (Ground Truth)
  plt.subplot(2, 3, 2)
  plt.imshow(img_display, cmap='gray')
  plt.imshow(label_display, cmap='jet', alpha=0.5)
  plt.title('Masque de vérité terrain')
  plt.axis('off')

  # Prédiction cas 1 baseline HR
  plt.subplot(2, 3, 3)
  plt.imshow(img_display, cmap='gray')
  plt.imshow(pred_model1_display, cmap='jet', alpha=0.5)
  plt.title('Prédiction du modèle 1')
  plt.axis('off')

  # Prédiction cas 2 Interpolation bilinéaire
  plt.subplot(2, 3, 4)
  plt.imshow(img_display, cmap='gray')
  plt.imshow(pred_model2_display, cmap='jet', alpha=0.5)
  plt.title('Prédiction du modèle 2')
  plt.axis('off')

  # Prédiction cas 3 SRCNN figé
  plt.subplot(2, 3, 5)
  plt.imshow(img_display, cmap='gray')
  plt.imshow(pred_model3_display, cmap='jet', alpha=0.5)
  plt.title('Prédiction du modèle 3')
  plt.axis('off')

  # Prédiction cas 4 U-Net + SRCNN conjoint
  plt.subplot(2, 3, 6)
  plt.imshow(img_display, cmap='gray')
  plt.imshow(pred_model4_display, cmap='jet', alpha=0.5)
  plt.title('Prédiction du modèle 4')
  plt.axis('off')

  plt.tight_layout()
  plt.savefig(os.path.join(base_projet, '2D', 'comparaison_visuelle_decathlon.png'), dpi=300, bbox_inches='tight')
  plt.show()

afficher_comparaison_visuelle(0)


# Calcul anomalies géométriques

In [ ]:
def calcul_circularite(masque_binaire):
  # Proche de 1 : sphérique/dilatée, proche de 0 : allongée
  aire = np.sum(masque_binaire)
  perimetre = perimeter(masque_binaire)
  if perimetre == 0:
    return 0
  circularite = 4 * np.pi * aire / (perimetre ** 2)
  return min(1,circularite)

def calcul_elongation(masque_binaire):
  # Proche de 0 : sphérique/dilatée, proche de 1 : allongée
  masque_label = label(masque_binaire.astype(bool))
  regions = regionprops(masque_label)
  if len(regions) == 0:
    return 1
  atrium_principal = max(regions, key=lambda x: x.area)
  return float(atrium_principal.eccentricity)

def generer_carte_ecarts_locaux(masque_expert, masque_predit):
  # Génération carte de distance euclidienne minimale entre fronitère du modèle et contour de l'expert
  masque_expert = masque_expert.astype(bool)
  masque_predit = masque_predit.astype(bool)

  contour_expert = masque_expert ^ binary_erosion(masque_expert)
  contour_predit = masque_predit ^ binary_erosion(masque_predit)

  carte_distance_expert = ndimage.distance_transform_edt(~contour_expert)
  carte_ecarts = np.zeros_like(masque_expert, dtype=float)
  carte_ecarts[contour_predit] = carte_distance_expert[contour_predit]
  carte_ecarts_visuelle = ndimage.maximum_filter(carte_ecarts, size=3)

  ecart_max = np.max(carte_ecarts) if np.sum(contour_predit) > 0 else 0
  return carte_ecarts_visuelle, ecart_max

def executer_diagnostic_anomalies(nb_patients):
    """
    Exécute le pipeline complet de détection d'anomalies morphologiques et affiche
    un rapport clinique comparatif détaillé avec graphiques.
    """
    unet_model4.eval()
    if srcnn_joint is not None: srcnn_joint.eval()

    # Utilisation du val_loader pour garantir les bons formats de batch
    val_iter = iter(val_loader)

    fig, axes = plt.subplots(nb_patients, 3, figsize=(10, 3*nb_patients), squeeze=False)
    im_heatmap = None

    with torch.no_grad():
      # Prédiction du modèle
      for i in range(min(nb_patients, len(val_dataset_2d))):
        try:
          patient = next(val_iter)
        except StopIteration:
          print(f"Fin du DataLoader atteinte à {i} patients")
          break

        img_lr = patient["image_lr"].unsqueeze(0).to(device)
        img_hr = patient["image_hr"].unsqueeze(0).to(device)
        label_gt = patient["label"].unsqueeze(0).to(device)

        # Inférence Cas 4 Conjoint
        img_upsampled = nn.functional.interpolate(img_lr, size=(256,256), mode='bilinear', align_corners=False)
        img_srcnn_joint = srcnn_joint(img_upsampled)
        output_model4 = unet_model4(img_srcnn_joint)
        bin_model4 = post_binarize(torch.sigmoid(output_model4))
        pred_model4 = keep_largest(bin_model4)

        # Reconversion en tableaux Numpy pour Scipy/Matplotlib
        label_display = label_gt[0, 0].cpu().numpy()
        pred_display = pred_model4[0, 0].cpu().numpy()

        # Calculs des métriques géométriques
        circ_expert = calcul_circularite(label_display)
        circ_predit = calcul_circularite(pred_display)

        ecc_expert = calcul_elongation(label_display)
        ecc_predit = calcul_elongation(pred_display)

        carte_ecarts, ecart_max = generer_carte_ecarts_locaux(label_display, pred_display)

        # Affichage console
        print(f"Patient {i+1}")
        print(f"Circ Expert: {circ_expert:.2f} / Circ Prédit: {circ_predit:.2f}")
        print(f"Delta Circ: {abs(circ_expert-circ_predit):.3f}")
        print(f"Ecc Expert: {ecc_expert:.2f} / Ecc Prédit: {ecc_predit:.2f}")
        print(f"Delta Ecc: {abs(ecc_expert-ecc_predit):.3f}")
        print(f"Ecart max de frontière : {ecart_max:.2f} pixels\n")

        # Visualisation des masques
        axes[i, 0].imshow(label_display, cmap='Blues')
        axes[i, 0].set_title(f"{i+1} Expert (Circ: {circ_expert:.2f} / Ecc: {ecc_expert:.2f})")
        axes[i, 0].axis('off')

        axes[i, 1].imshow(pred_display, cmap='Oranges')
        axes[i, 1].set_title(f"{i+1} Modèle (Circ: {circ_predit:.2f} / Ecc: {ecc_predit:.2f})")
        axes[i, 1].axis('off')

        im_heatmap = axes[i, 2].imshow(carte_ecarts, cmap='hot', vmin=0, vmax=15)
        axes[i, 2].set_title(f"Carte d'écarts locaux (max: {ecart_max:.2f})")
        axes[i, 2].axis('off')

    # Ajout de la colorbar
    if im_heatmap is not None:
      color_bar = plt.colorbar(im_heatmap, ax=axes[:, 2] fraction=0.02, pad=0.03)
      color_bar.set_label("Distance d'erreur locale(pixels)", weight='bold')

    # Ajustement automatique des espaces
    plt.tight_layout()

    # Sauvegarde
    plt.savefig(os.path.join(base_projet, '2D', 'diagnostic_anomalies_decathlon.png'), dpi=300, bbox_inches='tight')
    plt.show()

executer_diagnostic_anomalies(nb_patients=4)